# Urban ML Pipeline — Analysis

Este notebook analiza el dataset generado por `run_pipeline.py` y ejecuta todo el flujo de ML requerido por el curso DEML:

1. **EDA** — Distribución de clases, boxplots, correlaciones, SOM
2. **Reducción de dimensionalidad** — PCA (biplot), ICA (mixing matrix), t-SNE
3. **Experimentos de encoding** — Escalado (StandardScaler vs MinMaxScaler vs ninguno) y codificación del target
4. **Clasificación supervisada** — Logistic Regression (baseline), XGBoost, Random Forest, SVC, ANN
5. **Ablation study + tuning** — Quitar UNA feature a la vez, GridSearchCV
6. **Clustering no supervisado** — K-Means (elbow + silhouette)
7. **Heatmaps** — Visualización geográfica de predicciones + comparación entre ciudades

**Cómo usar:** Ejecuta `python run_pipeline.py` primero para generar `csv/all_cities_combined.csv`, luego corre este notebook celda por celda. Cada sección genera plots en `outputs/`.

### Setup — Importaciones y configuración

Carga todas las librerías necesarias. Los imports opcionales (xgboost, tensorflow, minisom, folium) están en bloques `try/except` — si no están instalados, el notebook sigue funcionando y simplemente salta esas secciones.

También importa las constantes del pipeline desde `config.py`: lista de features, mapeo de clases, configuración de la red neuronal, etc.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, FastICA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping XGBoost")

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("tensorflow not installed — skipping ANN")

try:
    from minisom import MiniSom
    HAS_SOM = True
except ImportError:
    HAS_SOM = False
    print("minisom not installed — run: pip install minisom")

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — skipping interactive maps")

# Import pipeline config
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from config import FEATURE_COLS, get_class_map, CITY_REGISTRY, ANN_CONFIG, KMEANS_MAX_K, TSNE_PERPLEXITY

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
os.makedirs("outputs", exist_ok=True)
print("Setup complete")

## 0. Data Loading

Carga el CSV combinado con todas las ciudades. Este archivo es generado por `run_pipeline.py` y contiene una fila por cada celda del grid (150m x 150m) con sus 10 features y la columna `zone_type` (la etiqueta Y).

**Qué buscar en el output:**
- Número total de filas (cada fila = una celda del grid)
- Lista de ciudades incluidas
- Nombres de las columnas disponibles

### Preparación del dataset

Aplica el mapeo de clases (`Commercial`, `Mixed-Use` → `Residential`, etc.), selecciona las 10 features, escala con `StandardScaler`, codifica las etiquetas y divide en train/test (80/20).

**Qué buscar en el output:**
- **Class distribution:** ¿Cuántas celdas hay por clase? Si hay mucho desbalance (>5:1), los modelos necesitan `class_weight="balanced"`.
- **Missing features:** Si aparece un WARNING, alguna feature no fue generada por la pipeline.
- **N_FOLDS:** Si la clase más pequeña tiene pocas muestras, el CV usa menos folds automáticamente.
- **Train/Test split:** ~80% train, ~20% test. El test set es **sagrado** — nunca entra al entrenamiento.

In [ ]:
DATA_PATH = "csv/all_cities_combined.csv"
assert os.path.exists(DATA_PATH), f"Run `python run_pipeline.py` first to generate {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Cities: {df_raw['city'].unique().tolist()}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Apply class mapping
class_map = get_class_map()
df = df_raw[df_raw["zone_type"].isin(class_map)].copy()
df["label"] = df["zone_type"].map(class_map)
print(f"After class mapping: {len(df)} rows")
print(f"Class distribution:\n{df['label'].value_counts()}")

# Feature matrix
available_features = [f for f in FEATURE_COLS if f in df.columns]
missing = [f for f in FEATURE_COLS if f not in df.columns]
if missing:
    print(f"WARNING: Missing features: {missing}")
print(f"Using {len(available_features)} features: {available_features}")

X = df[available_features].fillna(0).values
y = df["label"].values
cities = df["city"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_.tolist()
print(f"Classes: {class_names}")

# Determine safe number of CV folds based on smallest class
min_class_count = pd.Series(y_encoded).value_counts().min()
N_FOLDS = min(5, min_class_count)
if N_FOLDS < 5:
    print(f"WARNING: Smallest class has {min_class_count} samples — using {N_FOLDS}-fold CV instead of 5")

# Stratified split (need at least 2 per class)
if min_class_count >= 2:
    X_train, X_test, y_train, y_test, cities_train, cities_test = train_test_split(
        X_scaled, y_encoded, cities, test_size=0.2, random_state=42, stratify=y_encoded
    )
else:
    print("WARNING: Class too small for stratified split — using random split")
    X_train, X_test, y_train, y_test, cities_train, cities_test = train_test_split(
        X_scaled, y_encoded, cities, test_size=0.2, random_state=42
    )
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

### Boxplots — Distribución de cada feature por clase

Un boxplot por cada feature, separado por clase (Commercial vs Residential).

**Cómo leer un boxplot:**
- La **caja** contiene el 50% central de los datos (Q1 a Q3)
- La **línea verde** dentro de la caja = mediana
- Los **bigotes** se extienden hasta 1.5x el rango intercuartil
- Los **círculos** fuera de los bigotes = outliers

**Qué buscar:**
- Si las cajas de Commercial y Residential **NO se solapan** → esa feature separa bien las clases (buena feature)
- Si se solapan mucho → esa feature sola no distingue las clases (puede seguir siendo útil en combinación con otras)
- Muchos outliers → distribución sesgada, podría beneficiarse de log-transform

### Matriz de correlación

Muestra el coeficiente de Pearson (r) entre cada par de features. Valores de -1 a 1.

**Cómo interpretar:**
- **r > 0.7** (rojo fuerte): Features altamente correlacionadas → probablemente redundantes (miden lo mismo). Candidatas a eliminar una de las dos.
- **r ≈ 0** (blanco): Features independientes → cada una aporta información diferente. Ideal.
- **r < -0.7** (azul fuerte): Correlación negativa fuerte → también redundantes (una es el "inverso" de la otra).
- **En la diagonal** siempre hay 1.0 (cada feature está perfectamente correlacionada consigo misma).

**Regla:** Si ningún par supera r=0.7, no hay redundancia severa y puedes mantener todas las features.

## 1. Exploratory Data Analysis (EDA)

El EDA es el primer paso obligatorio antes de entrenar cualquier modelo. Su objetivo es **entender los datos visualmente** antes de tomar decisiones.

**Regla del profesor:** "Visualizar los datos ANTES de entrenar — plots, plots, plots."

En esta sección generamos:
- **Distribución de clases** — ¿Está balanceado el dataset? ¿Cuántas celdas hay por clase y por ciudad?
- **Boxplots por feature** — ¿Qué features separan bien las clases? ¿Cuáles se solapan?
- **Matriz de correlación** — ¿Hay features redundantes (r > 0.7)? ¿Cuáles son independientes?

In [ ]:
# Class distribution per city
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["label"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "coral", "green"][:len(class_names)])
axes[0].set_title("Overall Class Distribution")
axes[0].set_ylabel("Count")

ct = pd.crosstab(df["city"], df["label"])
ct.plot.bar(ax=axes[1], color=["steelblue", "coral", "green"][:len(class_names)])
axes[1].set_title("Class Distribution per City")
axes[1].set_ylabel("Count")
axes[1].legend(title="Class")

plt.tight_layout()
plt.savefig("outputs/01_class_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Feature boxplots
n_feat = len(available_features)
n_cols_plot = 3
n_rows_plot = (n_feat + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(15, 4 * n_rows_plot))
axes = axes.flatten()

for i, feat in enumerate(available_features):
    df.boxplot(column=feat, by="label", ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel("")

for i in range(n_feat, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Feature Distributions by Class", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("outputs/02_feature_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap
corr = df[available_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("outputs/03_correlation_heatmap.png", bbox_inches="tight")
plt.show()

### 1.1 Self-Organizing Map (SOM / Mapa de Kohonen)

Un SOM es una **red neuronal no supervisada** que proyecta datos de alta dimensión (10 features) a una grilla 2D. Cada neurona del mapa "absorbe" los datos más parecidos a ella. Datos similares terminan en neuronas vecinas.

**¿Para qué sirve?** Para ver si los datos tienen agrupaciones naturales SIN darle las etiquetas al modelo. Si el SOM agrupa los datos de manera similar a nuestras clases (Commercial vs Residential), eso confirma que las features son buenas.

**Cómo interpretar los 3 plots:**
- **U-Matrix (izquierda):** Las zonas oscuras son fronteras entre clusters. Si hay fronteras claras → hay grupos naturales bien definidos.
- **Por Zone Type (centro):** Los puntos coloreados por clase. Si los colores se separan en regiones distintas del mapa → las features capturan la diferencia real entre clases.
- **Por City (derecha):** Los puntos coloreados por ciudad. Si las ciudades se mezclan → los patrones urbanos son universales. Si se separan → cada ciudad tiene su propia "personalidad".

**Component Planes:** Un heatmap por feature. Muestra cómo varía cada feature en el mapa. Compara los patrones con la distribución de clases — si una feature tiene el mismo patrón que la separación de clases, es una buena feature.

Librería: `minisom` (pip install minisom). Requiere escalado min-max (aquí usamos StandardScaler que también funciona).

In [ ]:
if HAS_SOM:
    # Subsample for SOM (winner mapping is O(n) per sample)
    SOM_MAX = 8000
    if len(X_scaled) > SOM_MAX:
        rng_som = np.random.RandomState(42)
        som_idx = rng_som.choice(len(X_scaled), SOM_MAX, replace=False)
        X_som = X_scaled[som_idx]
        y_som = y[som_idx]
        cities_som = cities[som_idx]
        print(f"SOM: subsampled {SOM_MAX} from {len(X_scaled)}")
    else:
        X_som = X_scaled
        y_som = y
        cities_som = cities
        som_idx = np.arange(len(X_scaled))

    # Train SOM on scaled features
    som_x, som_y = 10, 10  # 10x10 grid
    som = MiniSom(som_x, som_y, X_som.shape[1],
                  sigma=1.5, learning_rate=0.5, random_seed=42)
    som.random_weights_init(X_som)
    som.train_random(X_som, num_iteration=5000)

    # Plot 1: U-Matrix (distance between neighboring neurons)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    umatrix = som.distance_map()
    axes[0].imshow(umatrix.T, cmap="bone_r", origin="lower")
    axes[0].set_title("SOM U-Matrix (dark = cluster boundary)")
    axes[0].set_xlabel("SOM X"); axes[0].set_ylabel("SOM Y")

    # Plot 2: SOM colored by zone type
    zone_colors_map = {"Commercial": "red", "Residential": "blue", "Other": "green"}
    for i, (x_val, label) in enumerate(zip(X_som, y_som)):
        w = som.winner(x_val)
        color = zone_colors_map.get(label, "gray")
        axes[1].plot(w[0] + np.random.uniform(-0.3, 0.3),
                     w[1] + np.random.uniform(-0.3, 0.3),
                     "o", color=color, markersize=2, alpha=0.4)
    axes[1].set_xlim(-0.5, som_x - 0.5)
    axes[1].set_ylim(-0.5, som_y - 0.5)
    axes[1].set_title("SOM — colored by Zone Type")
    for label, color in zone_colors_map.items():
        if label in class_names:
            axes[1].plot([], [], "o", color=color, label=label, markersize=8)
    axes[1].legend()

    # Plot 3: SOM colored by city
    city_colors = plt.cm.tab10(np.linspace(0, 1, len(df["city"].unique())))
    city_list_som = sorted(df["city"].unique())
    city_color_map = {c: city_colors[i] for i, c in enumerate(city_list_som)}
    for i, (x_val, city_name) in enumerate(zip(X_som, cities_som)):
        w = som.winner(x_val)
        axes[2].plot(w[0] + np.random.uniform(-0.3, 0.3),
                     w[1] + np.random.uniform(-0.3, 0.3),
                     "o", color=city_color_map[city_name], markersize=2, alpha=0.4)
    axes[2].set_xlim(-0.5, som_x - 0.5)
    axes[2].set_ylim(-0.5, som_y - 0.5)
    axes[2].set_title("SOM — colored by City")
    for c in city_list_som:
        axes[2].plot([], [], "o", color=city_color_map[c], label=c, markersize=8)
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig("outputs/03b_som.png", bbox_inches="tight")
    plt.show()

    # Component planes
    X_raw_som = X[som_idx] if len(X) > SOM_MAX else X
    n_feat = len(available_features)
    fig, axes = plt.subplots(2, (n_feat + 1) // 2, figsize=(4 * ((n_feat + 1) // 2), 8))
    axes = axes.flatten()
    for i, feat in enumerate(available_features):
        plane = np.zeros((som_x, som_y))
        count = np.zeros((som_x, som_y))
        for x_val, feat_val in zip(X_som, X_raw_som[:, i]):
            w = som.winner(x_val)
            plane[w] += feat_val
            count[w] += 1
        count[count == 0] = 1
        plane /= count
        axes[i].imshow(plane.T, cmap="viridis", origin="lower")
        axes[i].set_title(feat, fontsize=9)
    for i in range(n_feat, len(axes)):
        axes[i].set_visible(False)
    plt.suptitle("SOM Component Planes (feature intensity per neuron)", y=1.02)
    plt.tight_layout()
    plt.savefig("outputs/03c_som_components.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping SOM — install minisom: pip install minisom")

## 2. Feature Analysis — Reducción de Dimensionalidad

Nuestros datos viven en 10 dimensiones (10 features). El cerebro humano solo puede ver en 2D o 3D. Las técnicas de reducción de dimensionalidad **comprimen** esas 10 dimensiones a 2 para poder visualizarlas.

Tres técnicas complementarias:
- **PCA**: Encuentra las direcciones de máxima varianza. Es **determinístico** (siempre el mismo resultado). Genera un **biplot** con flechas que muestran qué features apuntan en qué dirección.
- **ICA**: Encuentra señales estadísticamente **independientes** escondidas en los datos. Útil para identificar qué features llevan la misma información subyacente.
- **t-SNE**: Preserva las relaciones de **vecindario** — puntos que son vecinos en 10D quedan vecinos en 2D. Es estocástico (hay que fijar semilla). **NO se puede usar para entrenar modelos** — solo visualización.

**Pregunta clave (la que hará el profesor):** "¿Cómo saben que las features que eligieron son las correctas, y qué evidencia visual/matemática tienen?"

La respuesta está en estos plots.

### 2.1 PCA (Principal Component Analysis)

PCA busca los **eigenvectors** de la matriz de covarianza — los ejes que capturan la mayor varianza posible.

**Cómo interpretar:**

*Plot izquierdo — Explained Variance:*
- Cada barra azul = cuánta varianza captura ese componente
- Línea roja = varianza acumulada
- La línea punteada gris marca el **95%** — cuántos componentes necesitas para representar casi toda la información
- Si 2-3 componentes bastan → los datos son de baja dimensionalidad real. Si necesitas 8+ → cada feature aporta información distinta

*Plot derecho — Biplot:*
- Los **puntos** son las celdas del grid, coloreados por clase
- Las **flechas rojas** son las features. Su dirección y longitud indican:
  - Flechas **paralelas** = features redundantes (miden lo mismo)
  - Flechas **perpendiculares** = features con información independiente
  - Flechas **opuestas** = features negativamente correlacionadas
- Si los colores (clases) se separan → las features son buenas para clasificar

**Requiere:** `StandardScaler` antes de PCA (obligatorio según el profesor).

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
exp_var = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance
axes[0].bar(range(1, len(exp_var)+1), exp_var, alpha=0.6, label="Individual")
axes[0].plot(range(1, len(exp_var)+1), np.cumsum(exp_var), "ro-", label="Cumulative")
axes[0].set_xlabel("Component"); axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("PCA Explained Variance"); axes[0].legend()
axes[0].axhline(y=0.95, color="gray", linestyle="--", alpha=0.5)

# Biplot (PC1 vs PC2)
for i, cls in enumerate(class_names):
    mask = y_encoded == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.3, s=10, label=cls)

loadings = pca.components_[:2].T
scale = 3
for j, feat in enumerate(available_features):
    axes[1].annotate("", xy=(loadings[j,0]*scale, loadings[j,1]*scale), xytext=(0,0),
                     arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
    axes[1].text(loadings[j,0]*scale*1.1, loadings[j,1]*scale*1.1, feat, fontsize=7, color="red")
axes[1].set_xlabel(f"PC1 ({exp_var[0]:.1%})"); axes[1].set_ylabel(f"PC2 ({exp_var[1]:.1%})")
axes[1].set_title("PCA Biplot"); axes[1].legend(markerscale=3)
plt.tight_layout()
plt.savefig("outputs/04_pca.png", bbox_inches="tight")
plt.show()

# How many components for 95%?
n_95 = np.argmax(np.cumsum(exp_var) >= 0.95) + 1
print(f"Components for 95% variance: {n_95} of {len(exp_var)}")
print(f"Top 3 components explain: {np.sum(exp_var[:3]):.1%}")

### 2.2 ICA (Independent Component Analysis)

ICA busca señales estadísticamente **independientes** escondidas en tus features. Mientras PCA busca direcciones de máxima varianza (correlación lineal), ICA busca fuentes verdaderamente independientes (independencia estadística).

**Analogía:** Imagina una fiesta con 5 conversaciones simultáneas y 10 micrófonos. PCA encontraría las direcciones más ruidosas. ICA encontraría las 5 conversaciones individuales.

**Cómo interpretar:**

*Plots izquierdo/centro — IC1 vs IC2:*
- Si las clases se separan en algún eje IC → ese componente independiente captura la diferencia entre zonas
- Si las ciudades se separan → hay señales específicas de cada ciudad

*Plot derecho — Mixing Matrix:*
- Cada celda muestra cuánto contribuye cada feature original a cada componente independiente
- Valores altos (rojo oscuro) → esa feature domina ese componente
- Si dos features tienen valores altos en el MISMO IC → comparten la misma fuente de información (potencialmente redundantes)
- Si cada feature domina un IC diferente → todas aportan información única

In [ ]:
n_components_ica = min(5, len(available_features))
ica = FastICA(n_components=n_components_ica, random_state=42, max_iter=1000)
X_ica = ica.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IC1 vs IC2 colored by zone type
for i, cls in enumerate(class_names):
    mask = y_encoded == i
    axes[0].scatter(X_ica[mask, 0], X_ica[mask, 1], alpha=0.3, s=10, label=cls)
axes[0].set_xlabel("IC1"); axes[0].set_ylabel("IC2")
axes[0].set_title("ICA — by Zone Type"); axes[0].legend(markerscale=3)

# IC1 vs IC2 colored by city
city_list_ica = sorted(df["city"].unique())
for city_name in city_list_ica:
    mask = cities == city_name
    axes[1].scatter(X_ica[mask, 0], X_ica[mask, 1], alpha=0.3, s=10, label=city_name)
axes[1].set_xlabel("IC1"); axes[1].set_ylabel("IC2")
axes[1].set_title("ICA — by City"); axes[1].legend(markerscale=3, fontsize=8)

# Mixing matrix heatmap — shows how each original feature contributes to each IC
mixing = ica.mixing_
axes[2].imshow(np.abs(mixing), cmap="YlOrRd", aspect="auto")
axes[2].set_xticks(range(n_components_ica))
axes[2].set_xticklabels([f"IC{i+1}" for i in range(n_components_ica)])
axes[2].set_yticks(range(len(available_features)))
axes[2].set_yticklabels(available_features, fontsize=8)
axes[2].set_title("ICA Mixing Matrix (feature contributions)")
for i in range(len(available_features)):
    for j in range(n_components_ica):
        axes[2].text(j, i, f"{mixing[i,j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig("outputs/05_ica.png", bbox_inches="tight")
plt.show()

# Identify redundant features (high correlation between ICA-reconstructed signals)
print("ICA component kurtosis (higher = more non-Gaussian = more informative):")
for i in range(n_components_ica):
    k = kurtosis(X_ica[:, i])
    print(f"  IC{i+1}: kurtosis = {k:.2f}")

### Target Encoding: Label vs One-Hot

Demuestra la diferencia conceptual entre las dos formas de codificar la variable Y (etiqueta):
- **Label Encoding** (y = 0, 1): Lo que usan sklearn y los modelos de árboles.
- **One-Hot Encoding** (y = [[1,0], [0,1]]): Lo que usan las redes neuronales con softmax en la capa de salida.

Para modelos de árboles (RF, XGBoost), One-Hot del target no aplica. Solo importa en la sección 4.5 (ANN).

### 2.3 t-SNE (t-distributed Stochastic Neighbor Embedding)

t-SNE es una proyección **no lineal** que preserva las relaciones de vecindario: puntos que son similares en 10 dimensiones quedarán cerca en 2D.

**Diferencias clave con PCA:**
- PCA es lineal y determinístico. t-SNE es no lineal y estocástico (por eso fijamos `random_state=42`)
- PCA preserva distancias globales. t-SNE preserva vecindarios locales
- PCA se puede usar para entrenar modelos. t-SNE **NUNCA** — solo para visualización
- El parámetro `perplexity` controla cuántos vecinos considerar (5-50, usamos 30)

**Cómo interpretar:**
- Si las clases forman **clusters separados** → las features son excelentes, el problema es "fácil"
- Si hay **solapamiento parcial** con tendencias → las features capturan la estructura pero el problema tiene zonas ambiguas
- Si es una **nube homogénea** sin estructura → las features no distinguen las clases
- Clusters aislados (grupos pequeños separados) → datos atípicos que vale la pena investigar

**Por ciudad:** Si las ciudades se mezclan, los patrones urbanos son universales. Si forman clusters separados, cada ciudad es un "mundo" diferente.

In [ ]:
# t-SNE (subsample for large datasets — t-SNE is O(n²))
TSNE_MAX = 8000
if len(X_scaled) > TSNE_MAX:
    rng_tsne = np.random.RandomState(42)
    tsne_idx = rng_tsne.choice(len(X_scaled), TSNE_MAX, replace=False)
    X_tsne_input = X_scaled[tsne_idx]
    y_tsne = y_encoded[tsne_idx]
    cities_tsne = cities[tsne_idx]
    print(f"t-SNE: subsampled {TSNE_MAX} from {len(X_scaled)}")
else:
    X_tsne_input = X_scaled
    y_tsne = y_encoded
    cities_tsne = cities

perplexity = min(TSNE_PERPLEXITY, len(X_tsne_input) - 1)
if perplexity < TSNE_PERPLEXITY:
    print(f"WARNING: Dataset too small for perplexity={TSNE_PERPLEXITY}, using {perplexity}")

tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_input)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, cls in enumerate(class_names):
    mask = y_tsne == i
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=10, label=cls)
axes[0].set_title("t-SNE — by Zone Type"); axes[0].legend(markerscale=3)

city_list_tsne = sorted(df["city"].unique())
for city_name in city_list_tsne:
    mask = cities_tsne == city_name
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=10, label=city_name)
axes[1].set_title("t-SNE — by City"); axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig("outputs/06_tsne.png", bbox_inches="tight")
plt.show()

## 3. Experimentos de Encoding

El **encoding** es cómo representas los datos numéricamente antes de dárselos al modelo. Dos decisiones clave:

**1. Escalado de features (X):**
- **StandardScaler**: Resta la media y divide por la desviación estándar → distribución con media=0, std=1. Recomendado para PCA y modelos lineales.
- **MinMaxScaler**: Comprime todo entre 0 y 1. Útil para redes neuronales y SOMs.
- **Sin escalar**: Datos crudos. Algunos modelos (árboles) no lo necesitan, pero otros (SVM, LR) sí.
- **Log + StandardScaler**: Aplica logaritmo primero (útil si los datos tienen distribuciones muy sesgadas con outliers extremos).

**2. Codificación del target (Y):**
- **Label Encoding**: Y = [0, 1] — números enteros. Usado por sklearn (LR, RF, SVM, XGBoost).
- **One-Hot Encoding**: Y = [[1,0], [0,1]] — vector binario. Usado por redes neuronales con softmax.

**Cómo interpretar:** El mejor escalado es el que da mayor accuracy en cross-validation. Si la diferencia es mínima (<2%), el modelo es robusto al escalado y puedes elegir el que tenga más sentido teórico.

In [ ]:
# Experiment: StandardScaler vs MinMaxScaler vs No Scaling
# Using Logistic Regression as baseline
encoding_results = {}

# 1. StandardScaler (mean=0, std=1)
pipe_std = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_std = cross_val_score(pipe_std, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["StandardScaler"] = scores_std.mean()

# 2. MinMaxScaler (0 to 1)
pipe_mm = Pipeline([("scaler", MinMaxScaler()),
                    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_mm = cross_val_score(pipe_mm, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["MinMaxScaler"] = scores_mm.mean()

# 3. No scaling (raw features)
pipe_raw = Pipeline([("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_raw = cross_val_score(pipe_raw, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["No Scaling"] = scores_raw.mean()

# 4. Log-transform + StandardScaler (for skewed features)
X_log = np.log1p(np.abs(X))  # log(1+|x|) handles zeros and negatives
pipe_log = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_log = cross_val_score(pipe_log, X_log, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["Log + StandardScaler"] = scores_log.mean()

print(f"Encoding/Scaling comparison (Logistic Regression, {N_FOLDS}-fold CV):")
for name, acc in sorted(encoding_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<25s} {acc:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
names = list(encoding_results.keys())
accs = [encoding_results[n] for n in names]
ax.bar(names, accs, color=["steelblue", "coral", "gray", "forestgreen"])
ax.set_ylim(min(accs) - 0.05, max(accs) + 0.05)
ax.set_ylabel(f"Accuracy ({N_FOLDS}-fold CV)")
ax.set_title("Effect of Feature Scaling Strategy")
for i, (n, a) in enumerate(zip(names, accs)):
    ax.text(i, a + 0.005, f"{a:.4f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/07_encoding_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Target encoding experiment: Label vs One-Hot
# With Random Forest (which handles both)
print("Target encoding comparison:")
print("  Label Encoding: y = [0, 1] — used by tree-based models and most sklearn classifiers")
print("  One-Hot Encoding: y = [[1,0], [0,1]] — used by neural networks (softmax output)")
print()

# Label encoding + RF
rf_label = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
scores_label = cross_val_score(rf_label, X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")

# One-Hot encoding doesn't apply to sklearn classifiers (they need integer labels)
# but it matters for neural networks. Demonstrate the conceptual difference:
print(f"  RF with Label Encoding:  {scores_label.mean():.4f} +/- {scores_label.std():.4f}")
print()
print("  Note: One-Hot encoding of the target is used in Section 4.5 (ANN)")
print("  where the output layer has one neuron per class with softmax activation.")
print("  Tree-based models (RF, XGBoost) always use integer label encoding internally.")

## 4. Clasificación Supervisada

Aquí entrenamos modelos que aprenden a predecir la clase (Commercial vs Residential) a partir de las features.

**Regla del profesor:** "Siempre empezar con el modelo más simple (Logistic Regression como baseline). Solo complejizar si los datos lo justifican."

**Modelos Shallow (pocos parámetros, rápidos, interpretables):**
- **Logistic Regression** — El baseline obligatorio. Frontera de decisión lineal.
- **Random Forest** — Ensemble de árboles de decisión. Captura no-linealidades.
- **XGBoost** — Gradient boosting. Generalmente el mejor shallow model.
- **SVC** — Support Vector Classification. Busca el hiperplano que maximiza el margen entre clases.

**Modelo Deep (muchos parámetros, aprende patrones complejos):**
- **ANN (Keras)** — Red neuronal artificial. Solo usar si los shallow no son suficientes.

**Regla heurística:** Número de parámetros entrenables del modelo ≤ 1/10 del tamaño del training set.

**Todos los modelos usan `class_weight="balanced"`** para compensar el desbalance de clases (hay más Residential que Commercial).

### 4.1 Logistic Regression (Baseline — Shallow)

La Logistic Regression es siempre el **primer modelo** que debes probar. Usa una función sigmoide para trazar una frontera de decisión **lineal** en el espacio de features.

**¿Por qué empezar aquí?** Porque establece un piso de rendimiento. Si LR da 80%, sabes que cualquier modelo más complejo debe superar eso para justificar su complejidad.

**Cómo interpretar la Confusion Matrix:**
- **Diagonal** (arriba-izq, abajo-der): predicciones correctas. Cuanto más azul, mejor.
- **Fuera de diagonal**: errores.
  - Arriba-derecha: Commercial predicho como Residential (**falso negativo** de Commercial)
  - Abajo-izquierda: Residential predicho como Commercial (**falso positivo** de Commercial)
- Con `class_weight="balanced"`, el modelo puede ser "agresivo" prediciendo la clase minoritaria, generando más falsos positivos.

**Cross-validation (CV):** Divide los datos en N partes, entrena con N-1 y testea con 1, repite N veces. Da una estimación más robusta que un solo train/test split.

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=class_names, ax=ax, cmap="Blues")
ax.set_title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig("outputs/08_lr_confusion.png", bbox_inches="tight")
plt.show()

cv_scores_lr = cross_val_score(lr, X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
print(f"Cross-val accuracy: {cv_scores_lr.mean():.3f} +/- {cv_scores_lr.std():.3f}")

### 4.2 XGBoost (Shallow, Gradient Boosting)

XGBoost construye árboles de decisión **secuencialmente**: cada nuevo árbol intenta corregir los errores del anterior. Es generalmente el mejor modelo shallow para datos tabulares.

**Cómo interpretar el Feature Importance:**
- La barra más larga = la feature que XGBoost usa más para tomar decisiones
- Features con importancia alta → son las que realmente discriminan entre clases
- Features con importancia baja → podrían ser ruido (candidatas para ablation study)
- Compara con la importancia de Random Forest — si ambos coinciden en las top features, la señal es robusta

In [ ]:
if HAS_XGB:
    xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                        random_state=42, eval_metric="mlogloss")
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)

    print("XGBoost Results:")
    print(classification_report(y_test, y_pred_xgb, target_names=class_names))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels=class_names, ax=axes[0], cmap="Blues")
    axes[0].set_title("XGBoost — Confusion Matrix")

    importances = xgb.feature_importances_
    idx = np.argsort(importances)
    axes[1].barh([available_features[i] for i in idx], importances[idx], color="steelblue")
    axes[1].set_title("XGBoost — Feature Importance")
    plt.tight_layout()
    plt.savefig("outputs/09_xgb_results.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping XGBoost (not installed)")
    y_pred_xgb = None

### 4.3 Random Forest (Shallow, Ensemble de Árboles)

Random Forest entrena muchos árboles de decisión **en paralelo**, cada uno con una muestra aleatoria de los datos y features. La predicción final es el voto mayoritario de todos los árboles.

**Ventajas:** Robusto a overfitting, da feature importance, no necesita escalado.

**Cómo interpretar el Feature Importance:**
- Mide cuánto reduce cada feature la impureza (Gini) promediada sobre todos los árboles
- Compara el ranking con XGBoost: si ambos coinciden en las top 3 features → señal robusta
- Si difieren mucho → hay interacciones complejas entre features que cada modelo captura diferente

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=class_names, ax=axes[0], cmap="Blues")
axes[0].set_title("Random Forest — Confusion Matrix")

importances = rf.feature_importances_
idx = np.argsort(importances)
axes[1].barh([available_features[i] for i in idx], importances[idx], color="forestgreen")
axes[1].set_title("Random Forest — Feature Importance")
plt.tight_layout()
plt.savefig("outputs/10_rf_results.png", bbox_inches="tight")
plt.show()

### 4.4 Support Vector Classification (SVC — Shallow)

SVC busca el **hiperplano** que maximiza la distancia (margen) entre las clases. El kernel transforma los datos a un espacio de mayor dimensión donde pueden ser linealmente separables.

**Kernels probados:**
- **linear**: Frontera de decisión recta. Similar a Logistic Regression.
- **rbf** (Radial Basis Function): Frontera curva. Captura no-linealidades moderadas.
- **poly** (Polinomial): Frontera polinomial. Captura interacciones entre features.

**Cómo interpretar:**
- Si el kernel lineal es el mejor → el problema es linealmente separable (raro)
- Si rbf o poly son mucho mejores → hay patrones no lineales importantes
- Compara los 3 confusion matrices: ¿cuál comete menos errores en la clase minoritaria?

### Exportar predicciones

Usa el mejor modelo (Random Forest) para predecir TODAS las celdas del dataset (no solo el test set). Guarda un CSV por ciudad con las predicciones, que será usado por las secciones de heatmap y comparación.

La accuracy reportada aquí es de **re-sustitución** (predicción sobre datos que el modelo ya vio durante entrenamiento), así que será más alta que la del test set. La accuracy real de generalización es la del test set reportada arriba.

In [ ]:
# SVC training (subsample for large datasets — SVC is O(n²))
SVC_TRAIN_MAX = 10000
if len(X_train) > SVC_TRAIN_MAX:
    rng_svc = np.random.RandomState(42)
    idx_svc = rng_svc.choice(len(X_train), SVC_TRAIN_MAX, replace=False)
    X_train_svc = X_train[idx_svc]
    y_train_svc = y_train[idx_svc]
    print(f"SVC: subsampled {SVC_TRAIN_MAX} from {len(X_train)} training samples")
else:
    X_train_svc = X_train
    y_train_svc = y_train

svc_results = {}
for kernel in ["rbf", "linear", "poly"]:
    svc = SVC(kernel=kernel, class_weight="balanced", random_state=42)
    svc.fit(X_train_svc, y_train_svc)
    y_pred_svc = svc.predict(X_test)
    acc = accuracy_score(y_test, y_pred_svc)
    svc_results[kernel] = {"accuracy": acc, "y_pred": y_pred_svc}
    print(f"SVC ({kernel}): accuracy = {acc:.3f}")

best_kernel = max(svc_results, key=lambda k: svc_results[k]["accuracy"])
print("Best kernel:", best_kernel)
print(classification_report(y_test, svc_results[best_kernel]["y_pred"],
                            target_names=class_names, zero_division=0))

# Confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, kernel in enumerate(["rbf", "linear", "poly"]):
    ConfusionMatrixDisplay.from_predictions(
        le.inverse_transform(y_test),
        le.inverse_transform(svc_results[kernel]["y_pred"]),
        display_labels=class_names, ax=axes[i], cmap="Blues"
    )
    axes[i].set_title(f"SVC ({kernel}) — acc={svc_results[kernel]['accuracy']:.3f}")
plt.suptitle("Support Vector Classification", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/11_svc_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/11_svc_results.png")

### 4.5 Red Neuronal Artificial — Deep Model (Keras/TensorFlow)

Una red neuronal con múltiples capas ocultas. Arquitectura:
- **Capa de entrada**: 10 neuronas (una por feature)
- **Capas ocultas**: 64 → 32 neuronas con activación ReLU + Dropout (20%)
- **Capa de salida**: softmax (una neurona por clase, probabilidades que suman 1)

**Cheat sheet del profesor:**

| Problema | Activación salida | Loss |
|----------|------------------|------|
| Clasificación binaria | sigmoid | binary_crossentropy |
| Multiclase | softmax | categorical_crossentropy |
| Regresión | linear/none | MSE |

**Cómo interpretar las curvas de entrenamiento:**
- Si train y val **bajan juntas** → el modelo aprende bien
- Si train baja pero val **sube** → **overfitting** (el modelo memoriza en vez de generalizar)
- Si ambas se estancan alto → **underfitting** (modelo muy simple o datos insuficientes)
- Dropout (20%) mata conexiones aleatorias durante entrenamiento para reducir overfitting

**Usa One-Hot encoding** para el target: Y = [[1,0], [0,1]] en vez de Y = [0, 1].

In [ ]:
if HAS_TF:
    ANN_MAX = 10000
    n_classes = len(class_names)

    if len(X_train) > ANN_MAX:
        rng_ann = np.random.RandomState(42)
        ann_idx = rng_ann.choice(len(X_train), ANN_MAX, replace=False)
        X_train_ann = X_train[ann_idx]
        y_train_ann = y_train[ann_idx]
        print(f"ANN: subsampled {ANN_MAX} from {len(X_train)} training samples")
    else:
        X_train_ann = X_train
        y_train_ann = y_train

    y_train_cat = keras.utils.to_categorical(y_train_ann, n_classes)
    y_test_cat = keras.utils.to_categorical(y_test, n_classes)

    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(X_train_ann.shape[1],)))
    for units in ANN_CONFIG["hidden_layers"]:
        model.add(keras.layers.Dense(units, activation="relu"))
        model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(n_classes, activation="softmax"))

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=ANN_CONFIG["learning_rate"]),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    print("ANN Architecture:")
    model.summary()

    ann_epochs = min(ANN_CONFIG["epochs"], 15)
    history = model.fit(X_train_ann, y_train_cat, epochs=ann_epochs,
                        validation_split=ANN_CONFIG["validation_split"],
                        batch_size=64, verbose=1)

    y_pred_ann = model.predict(X_test).argmax(axis=1)
    acc_ann = accuracy_score(y_test, y_pred_ann)
    print("ANN accuracy:", f"{acc_ann:.3f}")
    print(classification_report(y_test, y_pred_ann, target_names=class_names))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ann, display_labels=class_names, ax=axes[2], cmap="Blues")
    axes[2].set_title("ANN acc=" + f"{acc_ann:.3f}")
    plt.tight_layout()
    plt.savefig("outputs/12_ann_results.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping ANN (tensorflow not installed)")
    acc_ann = None


### 4.6 Comparación de Modelos

Tabla resumen con la accuracy de cada modelo en el test set.

**Cómo interpretar:**
- El modelo con mayor accuracy es el "ganador", pero la diferencia importa:
  - <2% de diferencia → son equivalentes, elige el más simple/interpretable
  - >5% de diferencia → el modelo más complejo se justifica
- Compara LR (baseline lineal) vs los demás: **el salto de accuracy** mide cuánta no-linealidad hay en tus datos
- Si todos los modelos dan ~mismo resultado → el techo está en los datos, no en el algoritmo

**Recuerda:** Accuracy > 80% es el objetivo mínimo del profesor. < 75% = el modelo no sirve.

In [ ]:
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    f"SVC ({best_kernel})": svc_results[best_kernel]["accuracy"],
}
if HAS_XGB and y_pred_xgb is not None:
    results["XGBoost"] = accuracy_score(y_test, y_pred_xgb)
if HAS_TF and acc_ann is not None:
    results["ANN (Keras)"] = acc_ann

df_results = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
df_results = df_results.sort_values("Accuracy", ascending=False)
print(df_results.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_results["Model"], df_results["Accuracy"], color="steelblue")
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_title("Model Comparison")
for i, (_, row) in enumerate(df_results.iterrows()):
    ax.text(row["Accuracy"] + 0.01, i, f"{row['Accuracy']:.3f}", va="center")
plt.tight_layout()
plt.savefig("outputs/13_model_comparison.png", bbox_inches="tight")
plt.show()

### Tuning de SVC + Comparación actualizada

Mismo proceso de GridSearchCV pero para SVC (kernel, C, gamma). Al final muestra una tabla comparativa actualizada que incluye tanto los modelos con hiperparámetros default como los tuneados.

**Cómo interpretar la tabla final:** Si "RF (tuned)" o "SVC (tuned)" superan claramente a sus versiones default, el tuning valió la pena. Si no, los defaults ya eran buenos.

In [ ]:
# Export predictions using best supervised model (RF by default)
best_model = rf
y_pred_all = best_model.predict(X_scaled)
df["predicted"] = le.inverse_transform(y_pred_all)
df["correct"] = df["predicted"] == df["label"]

for city_key in df["city"].unique():
    city_dir = f"csv/{city_key}"
    os.makedirs(city_dir, exist_ok=True)
    df_city = df[df["city"] == city_key].copy()
    pred_path = f"{city_dir}/07_predictions.csv"
    df_city.to_csv(pred_path, index=False, encoding="utf-8")
    acc = df_city["correct"].mean()
    print(f"{city_key}: {acc:.3f} accuracy ({len(df_city)} cells) -> {pred_path}")

## 5. Ablation Study & Hyperparameter Tuning

**Regla del profesor:** "Cambiar UNA sola variable a la vez. Nunca cambiar varias cosas y decir 'mejoró'."

Dos experimentos diferentes:

**Ablation Study (Sección 5.1):** Quitar UNA feature a la vez y medir cuánto cambia la accuracy.
- Si la accuracy **baja** mucho sin la feature → era importante
- Si la accuracy **sube** o no cambia → era ruido y puedes eliminarla
- Esto responde la pregunta del profesor: "¿Cómo saben que las features que eligieron son las correctas?"

**Hyperparameter Tuning (Sección 5.2):** Probar diferentes configuraciones del modelo con `GridSearchCV`.
- Cambia hiperparámetros (profundidad del árbol, número de estimadores, kernel, etc.)
- Usa cross-validation para cada combinación
- Reporta la mejor combinación y cuánto mejora sobre la configuración default

### 5.1 Feature Ablation Study

Proceso: para cada una de las 10 features, la eliminamos del dataset y reentrenamos el modelo (Random Forest con cross-validation). Medimos la accuracy sin esa feature y calculamos el "drop" respecto al baseline (todas las features).

**Cómo interpretar el gráfico:**
- **Barras rojas (positivas):** Quitar esa feature BAJA la accuracy → la feature es importante. Cuanto más larga la barra, más importante.
- **Barras verdes (negativas):** Quitar esa feature SUBE la accuracy → la feature era ruido o confundía al modelo. Candidata a ser eliminada.
- La feature con la barra roja más larga es la **MVP** (Most Valuable Predictor)
- Si muchas features son verdes → el modelo podría funcionar igual o mejor con menos features (modelo más simple = menos overfitting)

In [ ]:
# Ablation: remove one feature at a time, measure accuracy change
print("Feature Ablation Study (Random Forest)")
print("=" * 60)

# Baseline accuracy with all features
baseline_acc = cross_val_score(
    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy"
).mean()
print(f"Baseline (all {len(available_features)} features): {baseline_acc:.4f}\n")

ablation_results = {}
for feat_idx, feat_name in enumerate(available_features):
    # Remove this feature
    X_ablated = np.delete(X_scaled, feat_idx, axis=1)
    acc = cross_val_score(
        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
        X_ablated, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy"
    ).mean()
    drop = baseline_acc - acc
    ablation_results[feat_name] = {"accuracy": acc, "drop": drop}
    direction = "DROP" if drop > 0 else "GAIN"
    print(f"  Without {feat_name:<25s} acc={acc:.4f}  ({direction}: {abs(drop):.4f})")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
features_sorted = sorted(ablation_results, key=lambda f: ablation_results[f]["drop"], reverse=True)
drops = [ablation_results[f]["drop"] for f in features_sorted]
colors = ["red" if d > 0 else "green" for d in drops]
ax.barh(features_sorted, drops, color=colors, alpha=0.7)
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlabel("Accuracy Drop (positive = feature is important)")
ax.set_title("Feature Ablation — Impact on Accuracy")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("outputs/14_ablation_study.png", bbox_inches="tight")
plt.show()

print(f"\nMost important: {features_sorted[0]} (drop = {ablation_results[features_sorted[0]]['drop']:.4f})")
print(f"Least important: {features_sorted[-1]} (drop = {ablation_results[features_sorted[-1]]['drop']:.4f})")

### Mapas interactivos (Folium)

Genera un mapa HTML interactivo por ciudad usando Folium (basado en Leaflet.js). Puedes hacer click en cada celda para ver su ID, predicción y etiqueta real.

Los mapas se guardan en `outputs/{ciudad}/heatmap.html` — ábrelos en el navegador para explorarlos.

### 5.2 Hyperparameter Tuning (GridSearchCV)

`GridSearchCV` prueba **todas las combinaciones** de hiperparámetros que le des y reporta cuál funciona mejor.

**Para Random Forest se prueban:**
- `n_estimators`: [50, 100, 200] — cuántos árboles
- `max_depth`: [4, 8, 12, None] — profundidad máxima de cada árbol
- `min_samples_leaf`: [1, 3, 5] — mínimo de muestras por hoja

**Para SVC se prueban:**
- `kernel`: [rbf, linear, poly] — tipo de frontera de decisión
- `C`: [0.1, 1, 10] — fuerza de regularización (C alto = menos regularización)
- `gamma`: [scale, auto] — alcance de influencia de cada punto de soporte

**Cómo interpretar:**
- Si la mejora sobre el default es <1% → el default ya era bueno, no vale la pena complicarse
- Si la mejora es >3% → los hiperparámetros default estaban lejos del óptimo
- El gráfico de barras muestra las top 10 combinaciones — si están todas muy juntas, el modelo es robusto a los hiperparámetros

In [ ]:
# GridSearch for Random Forest
print("Hyperparameter Tuning — Random Forest (GridSearchCV)")
print("=" * 60)

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [4, 8, 12, None],
    "min_samples_leaf": [1, 3, 5],
}

cv_folds_tuning = min(3, N_FOLDS)
grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid_rf, cv=StratifiedKFold(cv_folds_tuning), scoring="accuracy",
    n_jobs=-1, verbose=0
)
grid_rf.fit(X_scaled, y_encoded)

print(f"Best params: {grid_rf.best_params_}")
print(f"Best CV accuracy: {grid_rf.best_score_:.4f}")
print(f"Default RF accuracy: {baseline_acc:.4f}")
print(f"Improvement: {grid_rf.best_score_ - baseline_acc:+.4f}")

# Visualize top 10 parameter combinations
results_df = pd.DataFrame(grid_rf.cv_results_)
results_df = results_df.sort_values("rank_test_score").head(10)
fig, ax = plt.subplots(figsize=(10, 5))
labels = [str(p) for p in results_df["params"]]
labels = [l.replace("'", "").replace("{", "").replace("}", "") for l in labels]
ax.barh(range(len(labels)), results_df["mean_test_score"], xerr=results_df["std_test_score"],
        color="steelblue", alpha=0.7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel("Mean CV Accuracy")
ax.set_title("Top 10 Hyperparameter Combinations (RF)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("outputs/15_tuning_rf.png", bbox_inches="tight")
plt.show()

### Comparación de feature means entre ciudades

Muestra los valores promedio de cada feature por ciudad en dos formatos:

**Plot izquierdo — Z-score normalizado:** Cada feature se normaliza (media global = 0, desviación = 1) para que todas sean comparables en la misma escala. Valores positivos = esa ciudad está por encima de la media global para esa feature. Valores negativos = por debajo.

**Plot derecho — Escala cruda:** Valores originales sin normalizar. Útil para ver magnitudes absolutas, pero features como `total_bldg_area` (~900K) dominan la escala y hacen invisibles a las demás.

**Cómo interpretar el z-score:** Si NYC tiene z=+2 en `shop_density_km2` y Chicago tiene z=-1, significa que NYC tiene significativamente más tiendas por km² que el promedio de las ciudades, mientras Chicago tiene menos.

In [ ]:
# GridSearch for SVC (subsampled — SVC is O(n²), too slow on full dataset)
print("Hyperparameter Tuning — SVC (GridSearchCV, subsampled)")
print("=" * 60)

SVC_SUBSAMPLE = 5000
if len(X_scaled) > SVC_SUBSAMPLE:
    rng = np.random.RandomState(42)
    idx_sub = rng.choice(len(X_scaled), SVC_SUBSAMPLE, replace=False)
    X_svc_tune = X_scaled[idx_sub]
    y_svc_tune = y_encoded[idx_sub]
    print(f"Subsampled {SVC_SUBSAMPLE} from {len(X_scaled)} for SVC tuning")
else:
    X_svc_tune = X_scaled
    y_svc_tune = y_encoded

param_grid_svc = {
    "kernel": ["rbf", "linear", "poly"],
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"],
}

grid_svc = GridSearchCV(
    SVC(class_weight="balanced", random_state=42),
    param_grid_svc, cv=StratifiedKFold(cv_folds_tuning), scoring="accuracy",
    n_jobs=-1, verbose=0
)
grid_svc.fit(X_svc_tune, y_svc_tune)

print(f"Best params: {grid_svc.best_params_}")
print(f"Best CV accuracy: {grid_svc.best_score_:.4f}")

# Update model comparison with tuned models
print("--- Updated Model Comparison (with tuning) ---")
tuned_results = dict(results)  # copy original results
tuned_results["RF (tuned)"] = grid_rf.best_score_
tuned_results["SVC (tuned)"] = grid_svc.best_score_

df_tuned = pd.DataFrame(list(tuned_results.items()), columns=["Model", "Accuracy"])
df_tuned = df_tuned.sort_values("Accuracy", ascending=False)
print(df_tuned.to_string(index=False))

## 4b. Comparación: Binary vs 3-Class Classification

Ejecuta la clasificación con dos estrategias de categorías para comparar:
1. **Binary (sin Mixed-Use):** Solo Commercial y Residential. Las celdas Mixed-Use se excluyen del training.
2. **3-Class:** Commercial, Mixed-Use, y Residential como clases separadas.

**¿Por qué comparar?** K-Means encontró 3 clusters naturales (k=3), lo que sugiere que Mixed-Use es una categoría real. Pero Mixed-Use es inherentemente difuso — el threshold de 40% para definirlo es arbitrario. Si binary da >3% más accuracy, las fronteras de decisión son más limpias sin Mixed-Use.

In [ ]:
# Binary vs 3-Class Classification Comparison
from config import CLASS_MAP_BINARY, CLASS_MAP_3CLASS

comparison_results = {}

for mode_name, class_map_mode in [("Binary (no Mixed-Use)", CLASS_MAP_BINARY), 
                                   ("3-Class (with Mixed-Use)", CLASS_MAP_3CLASS)]:
    # Apply class mapping
    df_mode = df_raw[df_raw["zone_type"].isin(class_map_mode)].copy()
    df_mode["label"] = df_mode["zone_type"].map(class_map_mode)
    
    if len(df_mode) < 20:
        print(f"  {mode_name}: Not enough data ({len(df_mode)} rows) — skipping")
        continue
    
    X_mode = df_mode[available_features].fillna(0).values
    y_mode = LabelEncoder().fit_transform(df_mode["label"].values)
    class_names_mode = sorted(df_mode["label"].unique())
    
    min_class = pd.Series(y_mode).value_counts().min()
    n_folds_mode = min(5, min_class)
    
    if min_class < 2:
        print(f"  {mode_name}: Smallest class has {min_class} samples — skipping")
        continue
    
    scaler_mode = StandardScaler()
    X_mode_scaled = scaler_mode.fit_transform(X_mode)
    
    # Train Random Forest with cross-validation
    rf_mode = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    scores = cross_val_score(rf_mode, X_mode_scaled, y_mode, 
                             cv=StratifiedKFold(n_folds_mode), scoring="accuracy")
    
    comparison_results[mode_name] = {
        "accuracy": scores.mean(),
        "std": scores.std(),
        "n_samples": len(df_mode),
        "n_classes": len(class_names_mode),
        "class_dist": df_mode["label"].value_counts().to_dict(),
    }
    
    print(f"  {mode_name}: accuracy = {scores.mean():.4f} +/- {scores.std():.4f} "
          f"({len(df_mode)} samples, {len(class_names_mode)} classes)")
    print(f"    Class distribution: {df_mode['label'].value_counts().to_dict()}")

# Plot comparison
if len(comparison_results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    modes = list(comparison_results.keys())
    accs = [comparison_results[m]["accuracy"] for m in modes]
    stds = [comparison_results[m]["std"] for m in modes]
    
    axes[0].bar(modes, accs, yerr=stds, color=["steelblue", "coral"], capsize=5)
    axes[0].set_ylabel("Accuracy (CV)")
    axes[0].set_title("Binary vs 3-Class Accuracy")
    axes[0].set_ylim(0, 1)
    for i, (m, a) in enumerate(zip(modes, accs)):
        axes[0].text(i, a + stds[i] + 0.02, f"{a:.3f}", ha="center")
    
    # Class distribution comparison
    for i, mode in enumerate(modes):
        dist = comparison_results[mode]["class_dist"]
        axes[1].bar([f"{mode}\n{k}" for k in dist.keys()], dist.values(),
                    color=["steelblue", "coral", "green"][:len(dist)])
    axes[1].set_title("Class Distribution per Mode")
    axes[1].set_ylabel("Count")
    
    plt.tight_layout()
    plt.savefig("outputs/20_binary_vs_3class.png", bbox_inches="tight")
    plt.show()
    
    # Determine winner
    best_mode = max(comparison_results, key=lambda m: comparison_results[m]["accuracy"])
    delta = abs(comparison_results[modes[0]]["accuracy"] - comparison_results[modes[1]]["accuracy"])
    print(f"\nBest mode: {best_mode}")
    print(f"Accuracy difference: {delta:.4f}")
    if delta < 0.02:
        print("Difference < 2% — modes are practically equivalent")
    elif delta < 0.05:
        print("Difference 2-5% — moderate advantage for the winner")
    else:
        print("Difference > 5% — significant advantage for the winner")

## 6. K-Means Clustering (No supervisado)

K-Means agrupa los datos en K clusters SIN usar las etiquetas. Esto verifica si la estructura natural de los datos coincide con nuestras clases.

**¿Para qué sirve según el profesor?** "Si K-Means separa las clases sin supervisión → las features son claras."

**Cómo interpretar:**

*Elbow Method (izquierda):*
- Inercia = suma de distancias al centroide. Baja siempre al aumentar K.
- El "codo" es donde la curva deja de bajar rápido → el K óptimo.
- Si no hay codo claro → los clusters no están bien definidos.

*Silhouette Score (derecha):*
- Mide qué tan bien separados están los clusters (-1 a 1). Mayor = mejor.
- El K con mayor silhouette = número natural de grupos en tus datos.
- Si el mejor K ≠ número de clases → puede haber subgrupos dentro de tus clases.

**Adjusted Rand Index (ARI):** Mide cuánto coinciden los clusters de K-Means con tus etiquetas reales. ARI=1 = coincidencia perfecta, ARI=0 = aleatorio.

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, KMEANS_MAX_K + 1)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_km = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels_km))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, "bo-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(list(K_range), sil_scores, "ro-")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
plt.tight_layout()
plt.savefig("outputs/16_kmeans_elbow.png", bbox_inches="tight")
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k}")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(X_scaled)
ari = adjusted_rand_score(y_encoded, cluster_labels)
print(f"Adjusted Rand Index (k={best_k} vs actual zones): {ari:.3f}")

ct = pd.crosstab(pd.Series(cluster_labels, name="Cluster"),
                 pd.Series(le.inverse_transform(y_encoded), name="Zone"))
print("\nCluster vs Zone Type:")
print(ct)

## 7. Heatmap — Visualización Geográfica

Representa las predicciones del modelo **espacialmente** sobre el mapa de cada ciudad. Cada punto es una celda del grid (150m x 150m) coloreada por la clase predicha.

**Paso 9 del proyecto:** "Representar evaluación gráficamente y geométricamente — heatmaps que muestren predicciones espacialmente."

**Cómo interpretar:**
- **Rojo = Commercial**, **Azul = Residential**
- Las zonas comerciales predichas deben coincidir con la realidad (distritos de negocios, centros comerciales)
- Las transiciones entre zonas deben ser graduales (no saltos aleatorios) — si son graduales, el modelo captura la estructura urbana real
- Outliers espaciales (un punto rojo rodeado de azules) merecen investigación: ¿error del modelo o caso real?

Se generan dos tipos:
1. **Matplotlib (estático)** — todos los cities en una fila, guardado como PNG
2. **Folium (interactivo)** — mapa web con click en cada celda para ver detalles, guardado como HTML

In [ ]:
HAS_COORDS = "cell_lat" in df.columns and "cell_lon" in df.columns

if HAS_COORDS:
    city_list = sorted(df["city"].unique())
    n_cities = len(city_list)

    fig, axes = plt.subplots(1, n_cities, figsize=(6 * n_cities, 6))
    if n_cities == 1:
        axes = [axes]

    zone_colors = {"Commercial": "red", "Residential": "blue", "Other": "green"}

    for i, city_key in enumerate(city_list):
        df_city = df[df["city"] == city_key]
        ax = axes[i]
        for label in df_city["predicted"].unique():
            mask = df_city["predicted"] == label
            color = zone_colors.get(label, "gray")
            ax.scatter(df_city.loc[mask, "cell_lon"], df_city.loc[mask, "cell_lat"],
                       c=color, s=3, alpha=0.5, label=label)
        ax.set_title(f"{city_key} ({len(df_city)} cells)")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.legend(markerscale=4, fontsize=8)
        ax.set_aspect("equal")

    plt.suptitle("Predicted Zone Types per City", fontsize=14)
    plt.tight_layout()
    plt.savefig("outputs/17_heatmap_all_cities.png", bbox_inches="tight")
    plt.show()
else:
    city_list = sorted(df["city"].unique())
    print("Skipping spatial heatmap — columns 'cell_lat'/'cell_lon' not found in dataset")
    print(f"Available columns: {list(df.columns)}")

In [ ]:
# Interactive folium maps (one per city)
if HAS_FOLIUM and HAS_COORDS:
    for city_key in city_list:
        df_city = df[df["city"] == city_key]
        center_lat = df_city["cell_lat"].mean()
        center_lon = df_city["cell_lon"].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

        color_map = {"Commercial": "red", "Residential": "blue", "Other": "green"}
        id_col = "cell_id" if "cell_id" in df_city.columns else None
        for _, row in df_city.iterrows():
            cell_label = f"{row[id_col]}: " if id_col else ""
            folium.CircleMarker(
                location=[row["cell_lat"], row["cell_lon"]],
                radius=3,
                color=color_map.get(row["predicted"], "gray"),
                fill=True, fill_opacity=0.6,
                popup=f"{cell_label}{row['predicted']} (actual: {row['label']})"
            ).add_to(m)

        os.makedirs(f"outputs/{city_key}", exist_ok=True)
        map_path = f"outputs/{city_key}/heatmap.html"
        m.save(map_path)
        print(f"Saved interactive map: {map_path}")
elif not HAS_COORDS:
    print("Skipping interactive maps — no coordinate columns")
else:
    print("Skipping interactive maps (folium not installed)")

## 8. Comparación entre Ciudades (Cross-City)

Compara el rendimiento del modelo y las características del dataset entre todas las ciudades incluidas.

**Tres gráficos:**
1. **Accuracy per City:** ¿En qué ciudad el modelo funciona mejor/peor? Ciudades con datos más limpios o patrones más claros tendrán mayor accuracy.
2. **Cell Count per City:** ¿Cuántas celdas aporta cada ciudad? Ciudades con más celdas dominan el entrenamiento.
3. **Class Balance per City:** ¿Es el desbalance Commercial/Residential consistente entre ciudades? Si una ciudad tiene 50/50 y otra 90/10, el modelo puede estar sesgado.

**Feature Means:** Compara los valores promedio de cada feature entre ciudades. Si `total_bldg_area` es 10x mayor en NYC que en Chicago → las ciudades tienen escalas muy diferentes, lo cual el StandardScaler maneja.

**Nota:** Con una sola ciudad estos plots son poco informativos. Se vuelven útiles cuando corres la pipeline con 2+ ciudades.

---

**Pipeline completa.** Todos los plots guardados en `outputs/`.

**Resumen de archivos generados:**
- `outputs/01-03` — EDA (distribución, boxplots, correlación)
- `outputs/03b-03c` — SOM (U-Matrix, component planes)
- `outputs/04-06` — Reducción dimensional (PCA, ICA, t-SNE)
- `outputs/07` — Experimento de encoding/scaling
- `outputs/08-12` — Modelos supervisados (LR, XGB, RF, SVC, ANN)
- `outputs/13` — Comparación de modelos
- `outputs/14-15` — Ablation study + hyperparameter tuning
- `outputs/16` — K-Means clustering
- `outputs/17` — Heatmap geográfico
- `outputs/18-19` — Comparación entre ciudades (feature means con z-score)
- `outputs/20` — Binary vs 3-Class classification comparison
- `outputs/21` — Transfer learning (Ground Truth vs OSM-Only)
- `outputs/{city}/heatmap.html` — Mapas interactivos Folium

In [ ]:
feat_means = df.groupby("city")[available_features].mean()
feat_means_norm = (feat_means - feat_means.mean()) / (feat_means.std() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Left: Normalized (z-score) — all features comparable
feat_means_norm.T.plot.bar(ax=axes[0])
axes[0].set_title("Feature Means per City (z-score normalized)")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Z-Score (0 = global mean)")
axes[0].legend(title="City", fontsize=8)
axes[0].axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].tick_params(axis="x", rotation=45)

# Right: Raw scale (reference)
feat_means.T.plot.bar(ax=axes[1])
axes[1].set_title("Feature Means per City (raw scale)")
axes[1].set_xlabel("Feature")
axes[1].set_ylabel("Mean Value")
axes[1].legend(title="City", fontsize=8)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("outputs/19_feature_means.png", bbox_inches="tight")
plt.show()

print("\nNormalized feature means per city (z-scores):")
print(feat_means_norm.round(2).to_string())

## 9. Transfer Learning: Ground Truth vs OSM-Only

**Pregunta central del proyecto:** ¿Puede un modelo entrenado con ciudades que tienen datos de propiedad (Ground Truth) predecir zonas en ciudades que solo tienen datos de OpenStreetMap?

**Diseño experimental:**
- **Grupo A (Ground Truth):** NYC, Philadelphia, Chicago — zone_type viene de PLUTO/OPA/Cook County
- **Grupo B (OSM-Only):** DC, SF, LA — zone_type viene de polígonos OSM landuse

**Proceso:**
1. Entrenar modelo SOLO con datos del Grupo A
2. Evaluar en test set del Grupo A (accuracy interna)
3. Predecir Grupo B con el modelo entrenado (zero-shot transfer)
4. Comparar accuracies → la diferencia mide cuánto se pierde al usar solo OSM

**Interpretación:**
- Si Grupo B accuracy > 80% → "OSM es suficiente para predecir zonas urbanas"
- Si Grupo B accuracy << Grupo A → "Los datos de propiedad aportan señal irreemplazable"

In [ ]:
# Transfer Learning: Train on Ground Truth, Predict OSM-Only
from config import CITY_REGISTRY

# Identify groups
gt_cities = [k for k, v in CITY_REGISTRY.items() if v.get("group") == "ground_truth"]
osm_cities = [k for k, v in CITY_REGISTRY.items() if v.get("group") == "osm_only"]

print(f"Ground Truth cities: {gt_cities}")
print(f"OSM-Only cities: {osm_cities}")

# Split data by group
df_gt = df[df["city"].isin(gt_cities)]
df_osm = df[df["city"].isin(osm_cities)]

print(f"\nGround Truth: {len(df_gt)} cells across {df_gt['city'].nunique()} cities")
print(f"OSM-Only: {len(df_osm)} cells across {df_osm['city'].nunique()} cities")

if len(df_gt) >= 20 and len(df_osm) >= 10:
    # Prepare Ground Truth data
    X_gt = df_gt[available_features].fillna(0).values
    y_gt = le.transform(df_gt["label"].values)
    
    scaler_gt = StandardScaler()
    X_gt_scaled = scaler_gt.fit_transform(X_gt)
    
    # Train/test split within Ground Truth
    X_gt_train, X_gt_test, y_gt_train, y_gt_test = train_test_split(
        X_gt_scaled, y_gt, test_size=0.2, random_state=42, 
        stratify=y_gt if pd.Series(y_gt).value_counts().min() >= 2 else None
    )
    
    # Train RF on Ground Truth
    rf_transfer = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    rf_transfer.fit(X_gt_train, y_gt_train)
    
    # Evaluate on Ground Truth test set
    acc_gt_test = accuracy_score(y_gt_test, rf_transfer.predict(X_gt_test))
    print(f"\nGround Truth test accuracy: {acc_gt_test:.3f}")
    
    # Predict OSM-Only (zero-shot transfer)
    X_osm = df_osm[available_features].fillna(0).values
    y_osm = le.transform(df_osm["label"].values)
    X_osm_scaled = scaler_gt.transform(X_osm)  # Use GT scaler!
    
    y_pred_osm = rf_transfer.predict(X_osm_scaled)
    acc_osm = accuracy_score(y_osm, y_pred_osm)
    print(f"OSM-Only transfer accuracy: {acc_osm:.3f}")
    print(f"Delta (GT - OSM): {acc_gt_test - acc_osm:+.3f}")
    
    # Per-city breakdown
    print("\nPer-city accuracy:")
    transfer_acc = {}
    for city in sorted(df["city"].unique()):
        df_city_tr = df[df["city"] == city]
        X_city = df_city_tr[available_features].fillna(0).values
        y_city = le.transform(df_city_tr["label"].values)
        X_city_scaled = scaler_gt.transform(X_city)
        y_pred_city = rf_transfer.predict(X_city_scaled)
        acc = accuracy_score(y_city, y_pred_city)
        group = "GT" if city in gt_cities else "OSM"
        transfer_acc[city] = {"accuracy": acc, "group": group, "n": len(df_city_tr)}
        print(f"  [{group}] {city:<15s} {acc:.3f}  ({len(df_city_tr)} cells)")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Plot 1: Accuracy by group
    acc_by_group = {"Ground Truth": acc_gt_test, "OSM-Only": acc_osm}
    colors_group = ["steelblue", "coral"]
    axes[0].bar(acc_by_group.keys(), acc_by_group.values(), color=colors_group)
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Ground Truth vs OSM-Only Accuracy")
    for i, (g, a) in enumerate(acc_by_group.items()):
        axes[0].text(i, a + 0.02, f"{a:.3f}", ha="center", fontsize=12, fontweight="bold")
    axes[0].axhline(y=0.8, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
    axes[0].legend()
    
    # Plot 2: Per-city accuracy colored by group
    cities_sorted_tr = sorted(transfer_acc, key=lambda c: transfer_acc[c]["accuracy"], reverse=True)
    city_accs = [transfer_acc[c]["accuracy"] for c in cities_sorted_tr]
    city_colors = ["steelblue" if transfer_acc[c]["group"] == "GT" else "coral" for c in cities_sorted_tr]
    axes[1].bar(cities_sorted_tr, city_accs, color=city_colors)
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Per-City Accuracy (trained on GT only)")
    for i, c in enumerate(cities_sorted_tr):
        axes[1].text(i, transfer_acc[c]["accuracy"] + 0.02, f"{transfer_acc[c]['accuracy']:.3f}", 
                     ha="center", fontsize=9)
    # Legend
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color="steelblue", label="Ground Truth"), 
                            Patch(color="coral", label="OSM-Only")], fontsize=9)
    
    # Plot 3: Confusion matrix for OSM-Only predictions
    ConfusionMatrixDisplay.from_predictions(y_osm, y_pred_osm, display_labels=class_names, 
                                            ax=axes[2], cmap="Blues")
    axes[2].set_title(f"OSM-Only Predictions (acc={acc_osm:.3f})")
    
    plt.tight_layout()
    plt.savefig("outputs/21_transfer_learning.png", bbox_inches="tight")
    plt.show()
    
    # Summary
    print(f"\n{'='*60}")
    if acc_osm >= 0.80:
        print("OSM-Only accuracy >= 80% — OSM data is sufficient for urban zone prediction")
    else:
        print("OSM-Only accuracy < 80% — property data provides irreplaceable signal")
    print(f"  Ground Truth: {acc_gt_test:.1%} | OSM-Only: {acc_osm:.1%} | Delta: {acc_gt_test-acc_osm:+.1%}")
else:
    if len(df_osm) < 10:
        print("Skipping transfer learning — no OSM-only cities in dataset")
        print(f"Run `python run_pipeline.py` with all 6 cities to enable this section")
    else:
        print("Skipping transfer learning — not enough Ground Truth data")

---

**Pipeline completa.** Todos los plots guardados en `outputs/`.

**Resumen de archivos generados:**
- `outputs/01-03` — EDA (distribución, boxplots, correlación)
- `outputs/03b-03c` — SOM (U-Matrix, component planes)
- `outputs/04-06` — Reducción dimensional (PCA, ICA, t-SNE)
- `outputs/07` — Experimento de encoding/scaling
- `outputs/08-12` — Modelos supervisados (LR, XGB, RF, SVC, ANN)
- `outputs/13` — Comparación de modelos
- `outputs/14-15` — Ablation study + hyperparameter tuning
- `outputs/16` — K-Means clustering
- `outputs/17` — Heatmap geográfico
- `outputs/18-19` — Comparación entre ciudades
- `outputs/{city}/heatmap.html` — Mapas interactivos Folium